# Redis Vector Store — Hands-On

**TechBot scenario:** Ingest product documentation into Redis, then retrieve the most relevant chunks when a user asks a question (RAG).

**Prerequisites:**
```bash
docker run -d --name redis-stack -p 6379:6379 -p 8001:8001 redis/redis-stack:latest
pip install redisvl sentence-transformers langchain-redis numpy
```

## Part 1: Create the Vector Index

In [ ]:
from redisvl.schema import IndexSchema
from redisvl.index import SearchIndex

# Define the schema for TechBot's knowledge base
# Each document chunk has: text, metadata fields, and an embedding vector
schema_dict = {
    "index": {
        "name": "rag-docs",
        "prefix": "doc",           # Redis keys will be: doc:0001, doc:0002, etc.
        "storage_type": "hash"     # store as Redis Hashes (simpler than JSON for this use case)
    },
    "fields": [
        # Full-text searchable — for keyword search fallback
        {"name": "text",     "type": "text"},
        # TAG fields — exact filter (use | for OR in queries)
        {"name": "source",   "type": "tag"},    # e.g. "manual", "api-ref", "blog"
        {"name": "category", "type": "tag"},    # e.g. "install", "auth", "billing"
        # NUMERIC field — range filter
        {"name": "page",     "type": "numeric"},
        # VECTOR field — the embedding
        {"name": "embedding", "type": "vector",
         "attrs": {
             "algorithm":       "hnsw",
             "dims":            384,          # all-MiniLM-L6-v2 output size
             "distance_metric": "cosine",     # best for text embeddings
             "datatype":        "float32",
             "m":               16,           # HNSW: graph connectivity
             "ef_construction": 200,          # HNSW: build quality
             "ef_runtime":      10            # HNSW: query recall vs speed
         }}
    ]
}

REDIS_URL = "redis://localhost:6379"

index = SearchIndex.from_dict(schema_dict, redis_url=REDIS_URL)
index.create(overwrite=True)  # overwrite=True: safe to re-run

print("Index created:", index.info()['index_name'])
print("Fields:", [f['identifier'] for f in index.info()['attributes']])

## Part 2: Load the Embedding Model

We use `all-MiniLM-L6-v2` — a fast, high-quality sentence embedding model that runs on CPU.
- 384 dimensions
- ~80MB download (cached after first run)
- No API key needed

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import time

# Downloads model on first run (~80MB), then uses local cache
print("Loading embedding model...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded.")

# Test it
test_embedding = embed_model.encode("Hello TechBot")
print(f"Embedding shape: {test_embedding.shape}")
print(f"Embedding dtype: {test_embedding.dtype}")
print(f"First 5 values: {test_embedding[:5]}")

## Part 3: Ingest TechBot Documentation

In [ ]:
# Simulated TechBot product documentation chunks
# In production: split real PDFs/markdown into ~500-word chunks using
# langchain_text_splitters.RecursiveCharacterTextSplitter

doc_chunks = [
    # Installation
    {"text": "To install the TechBot SDK, run: pip install techbot-sdk. Requires Python 3.9 or higher. After installation, verify with: python -c 'import techbot; print(techbot.__version__)'",
     "source": "manual", "category": "install", "page": 1},
    
    {"text": "On Windows, install the SDK using: pip install techbot-sdk. If you get a permission error, use: pip install --user techbot-sdk. For conda environments: conda install -c techcorp techbot-sdk",
     "source": "manual", "category": "install", "page": 2},
    
    {"text": "Docker installation: docker pull techcorp/techbot-sdk:latest. Run with: docker run -e TECHBOT_API_KEY=your_key techcorp/techbot-sdk:latest",
     "source": "manual", "category": "install", "page": 3},
    
    # Authentication
    {"text": "To authenticate with TechBot, set your API key as an environment variable: export TECHBOT_API_KEY=your_key_here. Then in Python: from techbot import Client; client = Client()  # auto-reads from environment",
     "source": "manual", "category": "auth", "page": 10},
    
    {"text": "Password reset: Go to https://app.techcorp.com/settings/security. Click 'Reset Password'. Enter your email address and check your inbox. Tokens expire after 24 hours.",
     "source": "manual", "category": "auth", "page": 11},
    
    {"text": "API key rotation: You can rotate your API key from the dashboard at Settings > API Keys > Rotate. Old keys are invalidated immediately. Update all services using the old key before rotating.",
     "source": "api-ref", "category": "auth", "page": 5},
    
    # Billing
    {"text": "Billing is monthly, charged on the 1st of each month. View your current usage and invoices at https://app.techcorp.com/billing. Usage is measured in API calls per month.",
     "source": "manual", "category": "billing", "page": 20},
    
    {"text": "The free tier allows 1,000 API calls per month. The Pro tier ($29/month) allows 100,000 calls. Enterprise plans are custom-priced. Overage is charged at $0.001 per call.",
     "source": "manual", "category": "billing", "page": 21},
    
    # Troubleshooting
    {"text": "If you get a 401 Unauthorized error, your API key is invalid or expired. Check that TECHBOT_API_KEY is set correctly and that the key has not been rotated or revoked.",
     "source": "manual", "category": "troubleshoot", "page": 30},
    
    {"text": "Rate limit errors (429 Too Many Requests) mean you've exceeded your plan's API call limit. Wait until the next billing cycle or upgrade your plan. Implement exponential backoff in your code.",
     "source": "manual", "category": "troubleshoot", "page": 31},
]

print(f"Preparing to ingest {len(doc_chunks)} document chunks...")

# Generate embeddings for all chunks
start = time.perf_counter()
texts = [d["text"] for d in doc_chunks]
embeddings = embed_model.encode(texts, batch_size=16, show_progress_bar=True)
embed_time = time.perf_counter() - start
print(f"Embedding time: {embed_time*1000:.0f}ms for {len(texts)} chunks")

# Attach embeddings to documents
# IMPORTANT: RedisVL expects embeddings as bytes for HASH storage
for i, (doc, emb) in enumerate(zip(doc_chunks, embeddings)):
    doc["embedding"] = np.array(emb, dtype=np.float32).tobytes()
    doc["id"] = f"{i+1:04d}"  # for reference

# Load into Redis
start = time.perf_counter()
keys = index.load(doc_chunks, id_field="id")
load_time = time.perf_counter() - start

print(f"\nLoaded {len(keys)} documents in {load_time*1000:.0f}ms")
print(f"Redis keys created: {keys[:3]}... (showing first 3)")

# Verify index
info = index.info()
print(f"\nIndex stats:")
print(f"  Documents indexed: {info['num_docs']}")
print(f"  Index name: {info['index_name']}")

## Part 4: Vector Similarity Search (KNN)

In [ ]:
from redisvl.query import VectorQuery

def search(query_text: str, top_k: int = 3, ef_runtime: int = 20) -> list:
    """Find the top_k most semantically similar documents."""
    query_vec = embed_model.encode(query_text).astype(np.float32).tolist()
    
    q = VectorQuery(
        vector=query_vec,
        vector_field_name="embedding",
        return_fields=["text", "source", "category", "page", "vector_distance"],
        num_results=top_k,
        ef_runtime=ef_runtime
    )
    
    results = index.query(q)
    return results

# Test query 1: Installation question
print("=" * 60)
print("Query: 'How do I install the SDK on Windows?'")
print("=" * 60)
results = search("How do I install the SDK on Windows?")
for r_item in results:
    dist = float(r_item['vector_distance'])
    print(f"\n  Distance: {dist:.4f} | Category: {r_item['category']} | Source: {r_item['source']}")
    print(f"  Text: {r_item['text'][:100]}...")

In [ ]:
# Test query 2: Password question
print("=" * 60)
print("Query: 'I forgot my password, what do I do?'")
print("=" * 60)
results = search("I forgot my password, what do I do?")
for r_item in results:
    dist = float(r_item['vector_distance'])
    print(f"\n  Distance: {dist:.4f} | Category: {r_item['category']}")
    print(f"  Text: {r_item['text'][:120]}...")

In [ ]:
# Test query 3: Billing question
print("=" * 60)
print("Query: 'How much does TechBot cost?'")
print("=" * 60)
results = search("How much does TechBot cost?")
for r_item in results:
    dist = float(r_item['vector_distance'])
    print(f"\n  Distance: {dist:.4f} | Category: {r_item['category']}")
    print(f"  Text: {r_item['text'][:120]}...")

## Part 5: Hybrid Search — Vector + Metadata Filter

In [ ]:
from redisvl.query import VectorQuery
from redisvl.query.filter import Tag, Num

def hybrid_search(query_text: str, category: str = None, source: str = None,
                  max_page: int = None, top_k: int = 3) -> list:
    """Vector search with optional metadata filters."""
    query_vec = embed_model.encode(query_text).astype(np.float32).tolist()
    
    # Build filter expression
    filters = []
    if category:
        filters.append(Tag("category") == category)
    if source:
        filters.append(Tag("source") == source)
    if max_page is not None:
        filters.append(Num("page") <= max_page)
    
    filter_expr = filters[0] if filters else None
    for f in filters[1:]:
        filter_expr = filter_expr & f
    
    q = VectorQuery(
        vector=query_vec,
        vector_field_name="embedding",
        filter_expression=filter_expr,
        return_fields=["text", "source", "category", "page", "vector_distance"],
        num_results=top_k
    )
    
    return index.query(q)

# Same query — but ONLY look in category=install
print("Hybrid search: 'How do I install?' AND category=install")
results = hybrid_search("How do I install?", category="install")
for r_item in results:
    print(f"  [{r_item['category']}] p{r_item['page']} dist={float(r_item['vector_distance']):.4f}: {r_item['text'][:80]}...")

print()

# Only look in source=api-ref
print("Hybrid search: 'authentication' AND source=api-ref")
results = hybrid_search("authentication", source="api-ref")
for r_item in results:
    print(f"  [{r_item['source']}] [{r_item['category']}] dist={float(r_item['vector_distance']):.4f}: {r_item['text'][:80]}...")

## Part 6: LangChain RAG Chain with Redis Vector Store

In [ ]:
from langchain_redis import RedisVectorStore
from langchain_core.documents import Document

# We need an embedding wrapper compatible with LangChain
# This wraps sentence-transformers so LangChain can use it
from langchain_community.embeddings import HuggingFaceEmbeddings

lc_embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

# Convert our doc_chunks to LangChain Document objects
lc_docs = [
    Document(
        page_content=chunk["text"],
        metadata={"source": chunk["source"], "category": chunk["category"], "page": chunk["page"]}
    )
    for chunk in doc_chunks
]

# Create vector store (overwrites existing index with same name)
lc_vectorstore = RedisVectorStore.from_documents(
    lc_docs,
    lc_embeddings,
    redis_url=REDIS_URL,
    index_name="langchain-rag",
)

print("LangChain vector store created.")

# Similarity search
results = lc_vectorstore.similarity_search_with_score(
    "How do I install the SDK?",
    k=3
)

print("\nLangChain similarity search results:")
for doc, score in results:
    print(f"  Score: {score:.4f} | Category: {doc.metadata['category']}")
    print(f"  Content: {doc.page_content[:80]}...")
    print()

In [ ]:
# Use Redis as a LangChain retriever for a RAG chain
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_core.messages import AIMessage

retriever = lc_vectorstore.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# Minimal RAG prompt
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are TechBot. Answer based ONLY on the context below. If the answer is not in the context, say so."),
    ("user", "Context:\n{context}\n\nQuestion: {question}")
])

# Fake LLM that prints the context (replace with real LLM in production)
def fake_rag_llm(messages):
    context_msg = messages.messages[1].content
    print("[What the LLM would see:]")
    print(context_msg[:400] + "..." if len(context_msg) > 400 else context_msg)
    return AIMessage(content="[Real LLM response would appear here based on the context above]")

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | RunnableLambda(fake_rag_llm)
    | StrOutputParser()
)

print("Running RAG chain for: 'How do I reset my password?'")
print("=" * 60)
response = rag_chain.invoke("How do I reset my password?")

## Part 7: Measure Latency

In [ ]:
import statistics

queries = [
    "How do I install?",
    "What's the price?",
    "I forgot my password",
    "Why am I getting 401 errors?",
    "How do I use Docker?"
]

latencies = []
for q in queries:
    start = time.perf_counter()
    results = search(q, top_k=3)
    latency_ms = (time.perf_counter() - start) * 1000
    latencies.append(latency_ms)
    print(f"  '{q[:40]}'  →  {latency_ms:.1f}ms  (top result: {results[0]['category'] if results else 'none'})")

print(f"\nP50 latency: {statistics.median(latencies):.1f}ms")
print(f"P95 latency: {sorted(latencies)[int(len(latencies)*0.95)]:.1f}ms")
print(f"Avg latency: {statistics.mean(latencies):.1f}ms")

## Cleanup

In [ ]:
# Drop indexes (this also deletes all indexed keys)
index.delete(drop=True)  # deletes rag-docs index + all doc:* keys
print("rag-docs index dropped.")

# Also clean up LangChain index
import redis as redis_lib
r_raw = redis_lib.Redis(host="localhost", port=6379, decode_responses=True)
lc_keys = list(r_raw.scan_iter("langchain-rag*"))
if lc_keys:
    r_raw.delete(*lc_keys)
print("LangChain index cleaned up.")

## Summary

You've built TechBot's RAG knowledge base:

1. **Created** a HNSW vector index with metadata fields
2. **Embedded** doc chunks using `all-MiniLM-L6-v2` (384-dim, free, local)
3. **Ingested** documents into Redis with `index.load()`
4. **Searched** with KNN `VectorQuery` — sub-5ms retrieval
5. **Filtered** with `Tag` and `Num` filters for hybrid search
6. **Integrated** with LangChain's `RedisVectorStore` and retriever

Next: **03_semantic_cache/02_hands_on.ipynb** — Avoid calling the LLM for repeated questions.